# KEEPCODING · AI Engineering · M6 n8n

## Clase 5 · Lab 1 · Tu primer workflow de n8n con AI Agent

En este laboratorio diseñas un **workflow de n8n** que orquesta un AI Agent con tools y lo deja listo para producción. Construimos el workflow en dos pasadas:

1. **Lo escribimos en Python** · emulando cada nodo, para entender qué hace n8n por dentro.
2. **Lo exportamos como `workflow.json`** · listo para importar a n8n (local con Docker o n8n Cloud) con un click.

El hilo conductor sigue siendo **One Piece**: el workflow recibe una pregunta vía webhook, deja decidir al AI Agent qué tool usar, llama a la API de One Piece y devuelve la respuesta formateada.

## Para quién es

Alumnado del curso que ya hizo M1 (agentes con LangChain/LangGraph) y quiere ver el mismo agente pero **construido visualmente con n8n**, no en código. No hace falta haber tocado n8n antes.

## Qué vas a aprender

- La **anatomía** de un workflow n8n: trigger, nodes, edges, items.
- El **AI Agent node** de n8n y sus sub-nodos: Chat Model, Tools, Memory.
- Cómo el **mismo agente** que escribirías en LangChain se cablea con drag-and-drop en n8n.
- Cómo **exportar** un workflow como JSON (versionable en git) e **importarlo**.
- Cómo arrancar n8n en local con **Docker** (1 comando) o usar **n8n Cloud**.
- Cómo invocar el workflow desde **fuera** (webhook HTTP).

## Conecta con

- **Slides M6** · sección "El nodo AI Agent" + "Triggers" + "Conexión con APIs externas".
- **M1 (Agentes con LangChain/LangGraph)** · este workflow hace lo mismo que el agente ReAct del lab de M1 pero sin escribir Python.
- **M3 (A2A)** · si publicas un workflow como webhook, otro agente puede consumirlo como tool A2A.
- **M5 (Harness)** · n8n es un harness no-code · cada sub-nodo del AI Agent es un pilar del modelo de M5.
- **M7-M9 · producción** (observabilidad, safety, coste y latencia) · n8n es un patrón canónico para desplegar workflows con IA sin gestionar infra.

## Stack

`requests` para llamar a APIs · `openai>=1.50` para emular el AI Agent node · n8n (Docker, `npx n8n` o n8n Cloud) para la parte visual.

## Hardware

CPU pura · ninguna GPU · todo vía API. n8n local pide ~500 MB RAM (Docker).


## Guion del laboratorio

1. **Setup**: instalar `requests` + `openai` y configurar la API key.
2. **Qué es un workflow n8n**: trigger, nodos, items[], JSON entre nodos.
3. **El AI Agent node**: el equivalente del `create_agent` de M1, pero visual.
4. **Diseñamos el workflow paso a paso**: trigger webhook + AI Agent + tool One Piece + respuesta.
5. **Emulamos el workflow en Python**: cada nodo es una función · vemos qué hace n8n por dentro.
6. **Exportamos `workflow.json`**: el fichero que importas a n8n con un click.
7. **Levantar n8n** (Docker, `npx n8n` o n8n Cloud) e importar el workflow.
8. **Probarlo desde Python**: `POST` al webhook con `requests`.
9. **Recap y retos**.

> Idea clave: el AI Agent de n8n es el mismo agente que el de M1, **pero cableado en vez de codificado**. Cuando entiendes lo que hace el código por debajo, leer un workflow de n8n se vuelve trivial.


## 1. Setup

### 1.1. Instalar dependencias

Para la parte de emulación necesitamos `requests` (HTTP client) y `openai` (SDK del modelo). Para la parte n8n no hace falta instalar nada Python · n8n se levanta con Docker.


In [1]:
# Instalación silenciosa
%pip install -q requests "openai>=1.50"
print("Dependencias instaladas.")


Dependencias instaladas.


### 1.2. Configurar `OPENAI_API_KEY`

El AI Agent node de n8n y nuestro emulador Python usan el mismo modelo (`gpt-5-mini`). En n8n configuras la credencial una vez en el panel y se cifra en la BD. En Python la cargamos en env.


In [2]:
import os
from getpass import getpass


def cargar_api_key() -> None:
    if os.environ.get("OPENAI_API_KEY"):
        print("OPENAI_API_KEY encontrada.")
        return
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or ""
        if os.environ["OPENAI_API_KEY"]:
            print("OPENAI_API_KEY cargada desde secrets de Colab.")
            return
    except Exception:
        pass
    try:
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY (vacío para saltar el paso 5): ")
    except Exception:
        pass


cargar_api_key()


OPENAI_API_KEY encontrada.


## 2. Qué es un workflow n8n

Un **workflow** es un grafo de nodos conectados que se dispara con un **trigger**. Cada nodo:

- Recibe `items[]` (lista de objetos JSON).
- Procesa cada item.
- Emite `items[]` al siguiente nodo.

Las conexiones entre nodos forman aristas: como un grafo dirigido. Las expresiones `{{ $json.campo }}` referencian datos del nodo anterior.

### Anatomía mínima

```
[Webhook Trigger]  ->  [AI Agent]  ->  [Respond to Webhook]
```

Eso es un workflow productivo de verdad: recibes una request HTTP, el agente piensa y actúa, devuelves la respuesta. Esto es lo que vamos a montar.

### Por qué importa el JSON

Cada workflow se exporta como un único JSON. Eso significa:
- Versionable en git ✓
- Reproducible entre entornos ✓
- Diffeable como código ✓

Es lo que rompe el mito de "low-code = no engineering": un workflow n8n es código (declarativo).


## 3. El AI Agent node

El **AI Agent node** es un nodo especial de n8n: por dentro es LangChain. Tiene la misma anatomía que el `create_agent` que ya conoces de M1, pero **sus partes se enchufan como sub-nodos**.

| Sub-nodo | Equivalente LangChain |
|---|---|
| **Chat Model** (sub-nodo) | el `ChatOpenAI(model=...)` |
| **Tools** (sub-nodos) | la lista `tools=[...]` |
| **Memory** (sub-nodo) | el `checkpointer=MemorySaver()` |
| **Output Parser** (sub-nodo opcional) | un `with_structured_output(...)` |

Al ver el workflow en n8n verás el AI Agent en el centro y, conectados por debajo, los sub-nodos. Cambiar el modelo es cambiar el sub-nodo Chat Model · no se toca el resto.

### Tipos de AI Agent

- **Tools Agent**: function calling moderno (lo que usa M1). **Default y recomendado**.
- **ReAct Agent**: el clásico con Thought/Action/Observation explícito.
- **Conversational Agent**: chatbot con historial.

Para este workflow usamos **Tools Agent**.


## 4. Diseñamos el workflow

El workflow tiene 4 nodos:

```
[Webhook Trigger]  ->  [AI Agent]  ->  [Respond to Webhook]
                            |
                            +---[Chat Model: OpenAI gpt-5-mini]
                            +---[Tool: HTTP Request -> One Piece API]
```

### Especificación

1. **Webhook Trigger** · path `/op-bounty` · método POST · espera body `{ "pregunta": "..." }`.
2. **AI Agent** · type `Tools Agent` · prompt de sistema descriptivo · acceso a 1 tool (HTTP Request).
3. **Tool · HTTP Request** · GET a `https://api.api-onepiece.com/v2/characters/en/search?name={nombre}` · devuelve recompensa.
4. **Respond to Webhook** · devuelve el `output` del agente como JSON.

> Si la API en vivo se cae, el AI Agent puede caer en un fallback (segunda tool con datos hardcoded). Para el lab inicial usamos solo la API real para que veas el patrón.


## 5. Emulamos el workflow en Python

Para que el lab funcione sin Docker, montamos el mismo workflow en Python. Cada nodo se convierte en una función · podemos ejecutar el workflow completo en una sola celda.

Esto es **didáctico**: te enseña qué pasa cuando un AI Agent node se dispara.


In [3]:
import json
import urllib.parse
import urllib.request
import unicodedata
from openai import OpenAI


# === Nodo "Tool: HTTP Request" === #
RECOMPENSAS_FALLBACK = {
    "luffy": ("Monkey D. Luffy", "3.000.000.000 berries", "Sombrero de Paja"),
    "zoro": ("Roronoa Zoro", "1.111.000.000 berries", "Sombrero de Paja"),
    "nami": ("Nami", "366.000.000 berries", "Sombrero de Paja"),
    "sanji": ("Vinsmoke Sanji", "1.032.000.000 berries", "Sombrero de Paja"),
    "robin": ("Nico Robin", "930.000.000 berries", "Sombrero de Paja"),
    "jinbe": ("Jinbe", "1.100.000.000 berries", "Sombrero de Paja"),
}


def _norm(t: str) -> str:
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode().lower()
    palabras = "".join(c if c.isalnum() else " " for c in t).split()
    return palabras[-1] if palabras else ""


def http_request_tool(nombre: str) -> dict:
    """Equivalente del nodo HTTP Request Tool en n8n.

    En n8n esto sería un sub-nodo del AI Agent configurado con:
      Method: GET
      URL: https://api.api-onepiece.com/v2/characters/en/search?name={{ $fromAI("nombre") }}

    Aquí lo emulamos: intentamos la API real y si falla, usamos el fallback.
    """
    try:
        url = f"https://api.api-onepiece.com/v2/characters/en/search?name={urllib.parse.quote(nombre)}"
        req = urllib.request.Request(url, headers={"User-Agent": "keepcoding-n8n-lab/1.0"})
        with urllib.request.urlopen(req, timeout=10) as r:
            data = json.load(r)
        if isinstance(data, list) and data:
            personaje = data[0]
            return {
                "nombre": personaje.get("name", nombre),
                "recompensa": personaje.get("bounty", "desconocida"),
                "tripulacion": personaje.get("crew", {}).get("name") if isinstance(personaje.get("crew"), dict) else "desconocida",
                "fuente": "api-onepiece.com",
            }
    except Exception:
        pass
    key = _norm(nombre)
    if key in RECOMPENSAS_FALLBACK:
        n, b, c = RECOMPENSAS_FALLBACK[key]
        return {"nombre": n, "recompensa": b, "tripulacion": c, "fuente": "fallback local"}
    return {"error": f"personaje no encontrado: {nombre}"}


print(http_request_tool("Luffy"))


{'nombre': 'Monkey D Luffy', 'recompensa': '3.000.000.000', 'tripulacion': 'The Chapeau de Paille crew', 'fuente': 'api-onepiece.com'}


In [4]:
# === Nodo "AI Agent" emulado === #
# En n8n esto es 1 nodo (AI Agent) con sub-nodos Chat Model y Tools.
# Aquí lo emulamos como un loop ReAct mínimo de OpenAI function-calling.

MODEL = "gpt-5-mini"
client = OpenAI() if os.environ.get("OPENAI_API_KEY") else None

TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "http_request_tool",
        "description": "Consulta la API de One Piece para obtener la recompensa de un personaje.",
        "parameters": {
            "type": "object",
            "properties": {"nombre": {"type": "string", "description": "Nombre del personaje"}},
            "required": ["nombre"],
        },
    },
}

SYSTEM_PROMPT_AI_AGENT = (
    "Eres un experto en One Piece. Cuando te pregunten por la recompensa "
    "(bounty) de un personaje, usa siempre la tool http_request_tool: "
    "no respondas de memoria. Formatea la cifra en berries con separadores "
    "de miles. Indica la fuente si está disponible."
)


def ai_agent_node(pregunta: str, max_iter: int = 4) -> str:
    """Emula el nodo AI Agent de n8n (type=Tools Agent)."""
    historial = [
        {"role": "system", "content": SYSTEM_PROMPT_AI_AGENT},
        {"role": "user", "content": pregunta},
    ]
    for _ in range(max_iter):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=historial,
            tools=[TOOL_SCHEMA],
        )
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content or ""
        historial.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            resultado = http_request_tool(**args)
            historial.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(resultado, ensure_ascii=False),
            })
    return "Límite de iteraciones alcanzado."


# === Nodo "Webhook Trigger" + "Respond to Webhook" emulados === #
def workflow(pregunta: str) -> dict:
    """Workflow completo: input -> AI Agent -> output.

    En n8n: [Webhook Trigger] -> [AI Agent] -> [Respond to Webhook].
    """
    respuesta = ai_agent_node(pregunta)
    return {"output": respuesta}


# Ejecutamos el workflow (necesita OPENAI_API_KEY)
if os.environ.get("OPENAI_API_KEY"):
    salida = workflow("¿Cuánto vale Sanji?")
    print("Workflow ejecutado.")
    print("Output:", salida["output"])
else:
    print("Sin OPENAI_API_KEY: salto la ejecución del workflow.")


Workflow ejecutado.
Output: La recompensa de Sanji es de 1.032.000.000 berries.  
Fuente: api-onepiece.com.


## 6. Exportar `workflow.json` (listo para importar a n8n)

Ahora viene la magia: ese mismo workflow lo describimos en un JSON con el schema de n8n. Lo guardamos al disco · es lo que importas con un click desde el panel n8n.

> En producción este `workflow.json` vive en git. Cuando hagas un cambio, lo importas como nueva versión.


In [5]:
WORKFLOW_JSON = {
    "name": "M6 · One Piece Bounty Agent",
    "nodes": [
        {
            "parameters": {
                "httpMethod": "POST",
                "path": "op-bounty",
                "responseMode": "responseNode",
                "options": {},
            },
            "id": "webhook-trigger",
            "name": "Webhook Trigger",
            "type": "n8n-nodes-base.webhook",
            "typeVersion": 2,
            "position": [240, 300],
            "webhookId": "op-bounty-webhook",
        },
        {
            "parameters": {
                "agent": "toolsAgent",
                "promptType": "define",
                "text": "={{ $json.body.pregunta }}",
                "options": {
                    "systemMessage": (
                        "Eres un experto en One Piece. Cuando te pregunten por "
                        "la recompensa (bounty) de un personaje, usa siempre la "
                        "tool http_request_tool: no respondas de memoria. "
                        "Formatea la cifra en berries con separadores de miles."
                    ),
                },
            },
            "id": "ai-agent",
            "name": "AI Agent",
            "type": "@n8n/n8n-nodes-langchain.agent",
            "typeVersion": 1.7,
            "position": [460, 300],
        },
        {
            "parameters": {
                "model": "gpt-5-mini",
                "options": {},
            },
            "id": "chat-model",
            "name": "OpenAI Chat Model",
            "type": "@n8n/n8n-nodes-langchain.lmChatOpenAi",
            "typeVersion": 1,
            "position": [400, 480],
            "credentials": {"openAiApi": {"id": "REPLACE_WITH_YOUR_CREDENTIAL_ID", "name": "OpenAI"}},
        },
        {
            "parameters": {
                "name": "http_request_tool",
                "description": "Consulta la API de One Piece. Param: nombre (string)",
                "method": "GET",
                "url": "=https://api.api-onepiece.com/v2/characters/en/search?name={{ $fromAI('nombre', 'Nombre del personaje', 'string') }}",
                "sendQuery": False,
                "options": {},
            },
            "id": "http-tool",
            "name": "HTTP Request Tool · One Piece",
            "type": "@n8n/n8n-nodes-langchain.toolHttpRequest",
            "typeVersion": 1.1,
            "position": [600, 480],
        },
        {
            "parameters": {
                "respondWith": "json",
                "responseBody": "={ \"output\": {{ JSON.stringify($json.output) }} }",
            },
            "id": "respond",
            "name": "Respond to Webhook",
            "type": "n8n-nodes-base.respondToWebhook",
            "typeVersion": 1,
            "position": [680, 300],
        },
    ],
    "connections": {
        "Webhook Trigger": {"main": [[{"node": "AI Agent", "type": "main", "index": 0}]]},
        "AI Agent": {"main": [[{"node": "Respond to Webhook", "type": "main", "index": 0}]]},
        "OpenAI Chat Model": {"ai_languageModel": [[{"node": "AI Agent", "type": "ai_languageModel", "index": 0}]]},
        "HTTP Request Tool · One Piece": {"ai_tool": [[{"node": "AI Agent", "type": "ai_tool", "index": 0}]]},
    },
    "settings": {"executionOrder": "v1"},
}

from pathlib import Path
out_path = Path("workflow.json")
out_path.write_text(json.dumps(WORKFLOW_JSON, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"workflow.json escrito ({out_path.stat().st_size} bytes)")
print("\nNodos definidos:")
for n in WORKFLOW_JSON["nodes"]:
    print(f"  - {n['name']} ({n['type']})")


workflow.json escrito (3254 bytes)

Nodos definidos:
  - Webhook Trigger (n8n-nodes-base.webhook)
  - AI Agent (@n8n/n8n-nodes-langchain.agent)
  - OpenAI Chat Model (@n8n/n8n-nodes-langchain.lmChatOpenAi)
  - HTTP Request Tool · One Piece (@n8n/n8n-nodes-langchain.toolHttpRequest)
  - Respond to Webhook (n8n-nodes-base.respondToWebhook)


### 6.1. Validamos el `workflow.json` sin n8n

Antes de importarlo, comprobamos offline que el JSON es coherente: hay un trigger, cada conexión apunta a un nodo que existe, el AI Agent tiene Chat Model y al menos una tool, y la credencial es un placeholder (nunca un ID real en git). Es el mismo tipo de check que pondrías en CI para un workflow versionado.


In [6]:
wf = json.loads(Path("workflow.json").read_text(encoding="utf-8"))
nombres = {nodo["name"] for nodo in wf["nodes"]}
tipos = {nodo["name"]: nodo["type"] for nodo in wf["nodes"]}
errores = []

# 1) Nombres de nodo únicos y un trigger webhook
if len(nombres) != len(wf["nodes"]):
    errores.append("Hay nodos con el nombre repetido.")
if "n8n-nodes-base.webhook" not in tipos.values():
    errores.append("Falta el Webhook Trigger.")

# 2) Todas las conexiones apuntan a nodos existentes
entradas_agente = []
for origen, salidas in wf["connections"].items():
    if origen not in nombres:
        errores.append(f"Conexión desde un nodo inexistente: {origen}")
    for tipo_con, ramas in salidas.items():
        for rama in ramas:
            for destino in rama:
                if destino["node"] not in nombres:
                    errores.append(f"Conexión hacia un nodo inexistente: {destino['node']}")
                if tipos.get(destino["node"]) == "@n8n/n8n-nodes-langchain.agent":
                    entradas_agente.append(tipo_con)

# 3) El AI Agent tiene Chat Model y al menos una tool enchufados
for requerido in ("ai_languageModel", "ai_tool"):
    if requerido not in entradas_agente:
        errores.append(f"El AI Agent no tiene ninguna conexión '{requerido}'.")

# 4) Credenciales: solo placeholders en el fichero versionado
for nodo in wf["nodes"]:
    for cred in nodo.get("credentials", {}).values():
        if not cred["id"].startswith("REPLACE_"):
            errores.append(f"El nodo '{nodo['name']}' lleva un credential ID real.")

print(f"Nodos: {len(wf['nodes'])} · conexiones de origen: {len(wf['connections'])}")
print("Entradas del AI Agent:", sorted(set(entradas_agente)))
print("workflow.json válido." if not errores else "Problemas encontrados:")
for e in errores:
    print("  -", e)


Nodos: 5 · conexiones de origen: 4
Entradas del AI Agent: ['ai_languageModel', 'ai_tool', 'main']
workflow.json válido.


## 7. Levantar n8n en local con Docker

Tienes el `workflow.json`. Ahora vamos a importarlo en n8n. Tienes tres caminos: Docker (1 comando), `npx n8n` (si tienes Node.js) o n8n Cloud (prueba gratuita).

### Opción A · Docker (recomendado)

Abre una terminal **fuera del notebook** y ejecuta:

```bash
docker run -it --rm \
  -p 5678:5678 \
  -v n8n_data:/home/node/.n8n \
  docker.n8n.io/n8nio/n8n
```

n8n estará en `http://localhost:5678`. La primera vez te pide crear cuenta admin (todo local).

### Importar el workflow

1. En el panel n8n, click en **"+"** arriba a la derecha · **"Import from File"**.
2. Selecciona el `workflow.json` que acabas de generar.
3. Verás los 4 nodos cableados como en el diagrama.

### Configurar credencial OpenAI

1. En el nodo `OpenAI Chat Model`, click en **"Credential"** · **"Create New Credential"**.
2. Pega tu `OPENAI_API_KEY`. Se cifra y se guarda en la BD de n8n.
3. Guarda y activa el workflow (toggle arriba a la derecha).

### Opción B · `npx n8n`

Si tienes Node.js instalado, `npx n8n` arranca la misma instancia en `http://localhost:5678` sin Docker.

### Opción C · n8n Cloud

Si no quieres instalar nada: `app.n8n.cloud` ofrece una prueba gratuita. Mismo proceso: importas el JSON y la URL del webhook será la de tu instancia cloud.


### Cómo se ve el workflow una vez importado

```
+--------------------+      +---------------+      +--------------------+
|  Webhook Trigger   | ---> |   AI Agent    | ---> | Respond to Webhook |
|  POST /op-bounty   |      |  Tools Agent  |      |   JSON output      |
+--------------------+      +-------+-------+      +--------------------+
                                    |
                            +-------+--------+
                            |                |
                    +-------v------+  +------v---------+
                    |  Chat Model  |  |   HTTP Tool    |
                    | gpt-5-mini   |  |  One Piece API |
                    +--------------+  +----------------+
```

Es el mismo agente que en M1 (`create_agent` + ChatOpenAI + tool con `@tool`), pero **cableado en lugar de codificado**. Quien entiende M1 lee este workflow en 30 segundos.


### Configurar la credencial OpenAI antes de importar

El `workflow.json` contiene un placeholder:

```json
"credentials": {
  "openAiApi": {
    "id": "REPLACE_WITH_YOUR_CREDENTIAL_ID",
    "name": "OpenAI"
  }
}
```

Tras importarlo a n8n:

1. Abre el workflow importado en n8n.
2. Click en el nodo `OpenAI Chat Model`.
3. En el panel derecho, sección **Credentials**, click en **Create New** o selecciona una existente.
4. Pega tu `OPENAI_API_KEY` en el campo correspondiente.
5. Guarda la credencial. n8n le asigna un ID interno (p. ej. `Wn4kQYx9...`) que sustituye al placeholder automáticamente al guardar el workflow.

El placeholder está pensado para que el `workflow.json` sea **versionable en git** sin filtrar IDs. Cada instancia de n8n (local, cloud, equipo) tiene sus propios credential IDs.

> Pro tip: si vas a versionar el workflow modificado, exporta otra vez como JSON tras configurar credenciales. n8n reemplaza la sección `credentials` con el ID real. **Sustituye ese ID por el placeholder antes de hacer commit** para no filtrar nada.


## 8. Probar el workflow desde Python

Una vez el workflow esté **activo** en tu n8n (toggle "Active"), el webhook estará en:

```
http://localhost:5678/webhook/op-bounty
```

(o `/webhook-test/op-bounty` en modo test desde el editor n8n; en n8n Cloud, la URL de tu instancia).

La celda lee la URL de la variable de entorno `N8N_WEBHOOK_URL`. Si no la defines, no intenta nada: te explica cómo levantar tu n8n y se salta la llamada, así el notebook corre entero sin n8n.


In [7]:
import requests

# URL del webhook de TU n8n. Ejemplo local: http://localhost:5678/webhook/op-bounty
N8N_WEBHOOK = os.environ.get("N8N_WEBHOOK_URL", "").strip()


def llamar_workflow(pregunta: str) -> dict | None:
    """POST al webhook del workflow n8n."""
    try:
        r = requests.post(N8N_WEBHOOK, json={"pregunta": pregunta}, timeout=60)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.ConnectionError:
        print(f"No hay n8n escuchando en {N8N_WEBHOOK}. Levántalo (paso 7) o usa el emulador del paso 5.")
    except requests.exceptions.HTTPError as e:
        print(f"Error HTTP: {e} · respuesta: {e.response.text[:200]}")
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"La llamada al webhook falló: {e}")
    return None


if N8N_WEBHOOK:
    respuesta = llamar_workflow("¿Cuánto vale Sanji?")
    print(respuesta)
else:
    print(
        "No hay N8N_WEBHOOK_URL definida: salto la llamada al webhook.\n"
        "Para probarlo con tu propio n8n:\n"
        "  1. Levanta n8n: `docker run -it --rm -p 5678:5678 docker.n8n.io/n8nio/n8n`,\n"
        "     o `npx n8n`, o abre una prueba gratuita en app.n8n.cloud.\n"
        "  2. Importa workflow.json, configura la credencial OpenAI y activa el workflow.\n"
        "  3. Define la URL del webhook, por ejemplo:\n"
        "     os.environ['N8N_WEBHOOK_URL'] = 'http://localhost:5678/webhook/op-bounty'\n"
        "  4. Vuelve a ejecutar esta celda.\n"
        "Mientras tanto, la emulación del paso 5 ya te muestra la misma respuesta."
    )


No hay N8N_WEBHOOK_URL definida: salto la llamada al webhook.
Para probarlo con tu propio n8n:
  1. Levanta n8n: `docker run -it --rm -p 5678:5678 docker.n8n.io/n8nio/n8n`,
     o `npx n8n`, o abre una prueba gratuita en app.n8n.cloud.
  2. Importa workflow.json, configura la credencial OpenAI y activa el workflow.
  3. Define la URL del webhook, por ejemplo:
     os.environ['N8N_WEBHOOK_URL'] = 'http://localhost:5678/webhook/op-bounty'
  4. Vuelve a ejecutar esta celda.
Mientras tanto, la emulación del paso 5 ya te muestra la misma respuesta.


## 9. Recap y retos

### Lo que has construido

- Un **workflow n8n** completo con AI Agent + tool One Piece + webhook trigger.
- La **emulación Python** del workflow · te demuestra que n8n no es magia: cada nodo es código que ya conoces.
- El `workflow.json` versionable en git · listo para importar a cualquier instancia de n8n.

### Ideas clave

- El AI Agent node de n8n **es el mismo agente que el de M1**, solo que cableado en vez de codificado.
- Un workflow n8n es **JSON declarativo**: versionable, diffeable, portable.
- Cambiar de modelo es cambiar 1 sub-nodo (Chat Model) sin tocar el resto.
- n8n es ideal para **prototipar rápido lo que en código tardarías mucho más** y para **conectar SaaS** (Slack, Notion, Stripe, GitHub) sin escribir clients.

### Retos para practicar

1. **Añade memoria**: enchufa un sub-nodo `Buffer Window Memory` al AI Agent, define `sessionId` desde el body del webhook. El agente recordará turnos previos por usuario.
2. **Añade otra tool**: un segundo HTTP Request Tool que consulte una API distinta (p. ej. PokéAPI). Que el agente decida cuál usar según la pregunta.
3. **Trigger Schedule**: cambia el Webhook Trigger por un Schedule Trigger que dispare cada lunes a las 9:00. El agente puede ser un "AI cronista" que escribe a Slack un resumen semanal.
4. **Human-in-the-Loop (2026)**: marca la HTTP tool como "requires approval" · n8n pide confirmación humana antes de ejecutarla. Útil para tools de impacto (escrituras en producción, enviar email).

### Conecta con

- **M1** (LangChain/LangGraph): el AI Agent node es `create_agent` cableado.
- **M3** (A2A): tu webhook puede ser el endpoint de un agente A2A · publica una Agent Card en `/.well-known/agent-card.json` con n8n y otros agentes te delegarán tareas.
- **M5** (Harness): n8n es un harness no-code. Cada sub-nodo del AI Agent corresponde a un pilar de M5.
- **M7-M9** (producción): observabilidad con Langfuse, safety y coste/latencia. n8n te da cron + webhooks + retry + secrets gestionados para desplegar workflows con IA.
